# Open-Fret: from MIDI to actual tablature

End-to-end demo using only repo code plus the released checkpoint:
build a MIDI phrase, transcribe it with the
[Hugging Face model](https://huggingface.co/mylonasc/open-fret-tab)
(`mylonasc/open-fret-tab`, tiny T5 trained on `vldsavelyev/guitar_tab` —
see the model card for training info, provenance and the unresolved
dataset-rights caveat), then render the **actual tablature** with the
repo's viewer (`open_fret.viewer`).

Provenance: the input below is a hand-written 8-note phrase, no corpus
material. The tab is a real model prediction through the repo's anchored
decoder (one fingering slot per input attack).

In [ ]:
import sys
sys.path.insert(0, "/home/charilaos/Workspace/open-fret")

import pretty_midi

# E-minor-pentatonic lick + closing dyad, ticks at 480 PPQ, 100 BPM.
# (start_tick, end_tick, midi_pitch)
phrase = [(0, 480, 64), (480, 960, 67), (960, 1440, 69), (1440, 1920, 71),
          (1920, 2400, 74), (2400, 2880, 76),
          (2880, 3360, 67), (2880, 3360, 64)]
SEC_PER_TICK = 60 / (100 * 480)
pm = pretty_midi.PrettyMIDI(initial_tempo=100)
inst = pretty_midi.Instrument(
    program=pretty_midi.instrument_name_to_program("Acoustic Guitar (nylon)"))
for start, end, pitch in phrase:
    inst.notes.append(pretty_midi.Note(velocity=100, pitch=pitch,
                                        start=start * SEC_PER_TICK,
                                        end=end * SEC_PER_TICK))
pm.instruments.append(inst)
MIDI_PATH = "/tmp/open-fret-notebook-demo.mid"
pm.write(MIDI_PATH)
print(f"wrote {MIDI_PATH}: {len(phrase)} notes,",
      f"pitches {sorted({p for _, _, p in phrase})}")

In [ ]:
import sys
sys.path.insert(0, "/home/charilaos/Workspace/open-fret/scripts")
from inference import run_inference

result = run_inference(MIDI_PATH, "mylonasc/open-fret-tab",
                        deterministic=True, return_report=True)
pred = result["corrected_tokens"]
print("predicted TAB tokens:", " ".join(pred))
print("corrections:", result["correction"]["refingered"], "refingered,",
      result["correction"]["inserted"], "inserted,",
      result["correction"]["dropped_extra"], "dropped")

In [ ]:
import sys
sys.path.insert(0, "/home/charilaos/Workspace/open-fret")
from open_fret.viewer import build_view_bundle, show

bundle = build_view_bundle(midi_path=MIDI_PATH, tab_tokens=pred,
                             title="Notebook demo — E-minor lick (model prediction)",
                             tempo=100)
print("bundle:", bundle["meta"]["n_input_notes"], "input notes,"
      , len(bundle["tab_tokens"]), "tab tokens")
show(bundle)  # <- the actual tablature: score + tab, playable, seekable

In [ ]:
import re
tuning = [64, 59, 55, 50, 45, 40]  # standard, high string first
input_pitches = {p for _, _, p in phrase}
tab = [t for t in pred if t.startswith("TAB_")]
assert tab, "prediction contains no TAB attacks"
bad = []
for t in tab:
    _, s, f = t.split("_")[:3]
    if tuning[int(s) - 1] + int(f) not in input_pitches:
        bad.append(t)
assert not bad, f"pitches outside the input: {bad}"
print(f"OK: {len(tab)} TAB attacks, every pitch present in the input {sorted(input_pitches)}")